# 生成天气图表

In [1]:
# 依赖包
import pandas as pd
from src import charts, om_api
from src import info_aure as aure
import matplotlib

# 基础数据
countries = aure.countries # 涉及的国家、地区
daily_indicators = aure.aure_daily_indicators  #需要查询的指标

# 存档的历史数据
au_path = ['dataset/AU_2024.csv',
           'dataset/AU_202508.csv']
eu_path = ['dataset/EU_2024.csv',
           'dataset/EU_202508.csv']
ru_path = ['dataset/RU_2024.csv',
           'dataset/RU_202508.csv']
ua_path = ['dataset/UA_2024.csv',
           'dataset/AU_202508.csv']
countries[0]['csv_path'] = au_path
countries[1]['csv_path'] = eu_path
countries[2]['csv_path'] = ru_path
countries[3]['csv_path'] = ua_path

# 时间区间
api_start_date      = '2025-09-01'
api_end_date        = '2025-09-02'
forecast_start_date = '2025-09-02'
forecast_end_date   = '2025-09-17'

for contry in countries:

    # 读取 CSV 历史数据（全部）
    csv_df = pd.DataFrame()
    for path in contry['csv_path']:
        df = pd.read_csv(path)
        csv_df = pd.concat([csv_df, df])
    csv_df['date'] = pd.to_datetime(csv_df['date'])

    for city in contry['city_list']:
        # 读取 CSV 历史数据（特定城市）
        city_csv_df = csv_df[csv_df['name'] == city['name']]

        # 请求 API 历史数据
        api_history_df = om_api.daily_history(city['latitude'], city['longitude'],
                                              api_start_date, api_end_date,
                                              daily_indicators)
        # 请求 API 预测数据
        api_forecast_df = om_api.daily_forecast(city['latitude'], city['longitude'],
                                                forecast_start_date, forecast_end_date,
                                                daily_indicators)
        # 合并数据
        concat_df = pd.concat([city_csv_df, api_history_df, api_forecast_df])
        concat_df = concat_df.groupby('date', as_index=False)[daily_indicators].mean()
        concat_df['name'] = city['name']

        # 数据处理
        concat_df = aure.data_prapare(concat_df)

        # 制图
        for style in aure.aure_styles:
            chart_params = {
                'min_history_year' : style['min_history_year'],
                'forecast_after' : forecast_start_date,
                'ylabel': style['ylabel'],
                'title' : style['title'] + city['name'],
            }
            chart = charts.day_annul_plot(concat_df, style['column'], **chart_params)
            path = f'./diagram/{contry['code']}/{contry['code']}{city['code']}_{style['path']}.jpg'
            chart.savefig(path, dpi=300)
            matplotlib.pyplot.close()

# 合并图片

# 保存历史数据

In [ ]:
import pandas as pd
from src import om_api
from src import info_aure as aure

# 确定需要存档的「时间范围」与「文件后缀」即可运行
start_date = '2024-01-01'
end_date = '2024-12-31'
tial_of_file = '2024'

for country in aure.countries:
    all_df = pd.DataFrame()
    for city  in country['city_list']:
        params = {
            'latitude'  : city['latitude'],
            'longitude' : city['longitude'],
            'start_date': start_date,
            'end_date'  : end_date,
            'daily_indicators': daily_indicators
        }
        api_df = om_api.daily_history(**params)
        groupby_df = api_df.groupby('date', as_index=False)[daily_indicators].mean()
        groupby_df['name'] = city['name']
        print('.. Loading: ' + city['name'])
        all_df = pd.concat([all_df, groupby_df])
    all_df.to_csv(f'dataset/{country['code']}_{tial_of_file}.csv', index=False)
    print('Finished: ' + country['name'])

# 合并历史数据文件

In [ ]:
import pandas as pd
from src import info_aure as aure

tial_of_input_file = ['2024','202508']
tial_of_output_file = 'xxx'

for country in aure.countries:
    out_df = pd.DataFrame()
    for t in tial_of_input_file:
        in_df = pd.read_csv(f'dataset/{country['code']}_{t}.csv', sep = ',')
        out_df = pd.concat([out_df, in_df])
    out_df.to_csv(f'dataset/{country["code"]}_{tial_of_output_file}.csv', index=False)